# Assignment 03: Parsing & Grammar Analysis

1. Create an ambiguous English sentence with at least two meanings.
แต่งประโยคภาษาอังกฤษที่มีความกำกวม (แปลได้มากกว่า 1 ความหมาย) มา 1 ประโยค

2. Write a CFG that accounts for all interpretations of the sentence.
เขียนกฎ CFG รองรับประโยคที่แต่งขึ้นมาให้ครอบคลุมทุกความหมาย

3. Parse the sentence and output all trees using NLTK’s ChartParser.
วิเคราะห์ประโยคและปรินต์ syntax trees ทั้งหมดออกมาด้วย ChartParser ของ NLTK

4. Define a PCFG with assumed rule probabilities.
กำหนดน้ำหนักความน่าจะเป็นลงในกฎ เพื่อสร้างเป็น PCFG (สมมุติตัวเลขความน่าจะเป็นเองได้เลย ไม่ต้องใช้ Treebank )

5. Print the most probable parse tree and its probability using ViterbiParser.
หา syntax trees ที่มีโอกาสเกิดมากที่สุดพร้อมค่าความน่าจะเป็นด้วย ViterbiParser ของ NLTK

6. Display the dependency parse tree of the sentence.
แสดงผลแผนภาพ Dependency Tree ของประโยคนั้น






In [1]:
# TODO: import lib here
import random
import spacy
import nltk

from nltk import CFG, PCFG, Nonterminal, Tree, induce_pcfg
from nltk.grammar import Production
from nltk.parse import ChartParser, ViterbiParser
from spacy import displacy

random.seed(13)
nlp = spacy.load("en_core_web_sm")

print("nltk  :", nltk.__version__)
print("spaCy :", spacy.__version__)

nltk  : 3.10.3
spaCy : 3.8.16


In [2]:
## TODO: 1
SENT = "I saw the man with the telescope."

In [3]:
## TODO: 2

grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP | 'I'
VP -> V NP | VP PP
PP -> P NP
Det -> 'the' | 'a'
N -> 'man' | 'telescope'
V -> 'saw'
P -> 'with'
""")
parser = ChartParser(grammar)
trees = trees = list(parser.parse(SENT.rstrip(".").split()))

In [4]:
## TODO: 3

print("number of possible trees:", len(trees))
print()

## HINT: for loop print all possible trees

number of possible trees: 2



In [5]:
## TODO: 4
PG = PCFG.fromstring("""
S -> NP VP [1.0]
NP -> Det N [0.5] | Det N PP [0.3] | 'I' [0.2]
VP -> V NP [0.6] | VP PP [0.4]
PP -> P NP [1.0]
Det -> 'the' [0.6] | 'a' [0.4]
N -> 'man' [0.7] | 'telescope' [0.3]
V -> 'saw' [1.0]
P -> 'with' [1.0]
""")

## check this first!
print("in CNF:", PG.is_chomsky_normal_form())
for lhs in [Nonterminal("NP"), Nonterminal("VP")]:
    ps = PG.productions(lhs=lhs)
    print(f"{lhs} probabilities sum to", round(sum(p.prob() for p in ps), 6))

in CNF: False
NP probabilities sum to 1.0
VP probabilities sum to 1.0


In [6]:
## TODO: 5
vp_parser = ViterbiParser(PG)
best_path = next(vp_parser.parse(SENT.rstrip(".").split()))
best_prob = best_path.prob()
print("best path probability:", best_prob)
best_path.pretty_print()

best path probability: 0.0013608
     S                                
  ___|_______                          
 |           VP                       
 |    _______|___                      
 |   |           NP                   
 |   |    _______|____                 
 |   |   |   |        PP              
 |   |   |   |    ____|___             
 |   |   |   |   |        NP          
 |   |   |   |   |     ___|______      
 NP  V  Det  N   P   Det         N    
 |   |   |   |   |    |          |     
 I  saw the man with the     telescope



In [7]:
## TODO: 6
doc = nlp(SENT)

displacy.render(doc, style="dep", jupyter=True,
                options={"compact": True, "distance": 95})